# Named Entity Recognition

Extracts named entities from every speech with spaCy (`en_core_web_sm`). Moved out of `03_feature_engineering.ipynb` so that re-running feature engineering does not repeat the slowest step of the pipeline.

**Caching:** extraction runs only when `RUN_NER = True` or when `ungdc_entities.parquet` does not exist; otherwise the saved results are loaded. NER uses `raw_text` (spaCy relies on capitalization), which is not affected by the normalization in notebook 02, so cached results remain valid after re-running preprocessing.

**Entity labels kept:** GPE (countries, cities), LOC (geographic locations), ORG (organizations), PERSON, NORP (nationalities, religious/political groups).

**Known limitations (to address in entity normalization):**
- The same entity appears under many surface forms (*the United States*, *United States*, *the United States of America*, *The United States*…)
- Some strings contain line breaks (*the United\nNations*)
- Systematic mislabels: *States* (from *Member States*) tagged GPE; *State*, *Powers*, *Treaty* tagged ORG
- `en_core_web_sm` is the smallest English model; recall on historical names is limited

## Setup

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ENTITIES_PATH = PROCESSED_DIR / "ungdc_entities.parquet"

# Set to True to re-run spaCy extraction (slow; requires en_core_web_sm)
RUN_NER = False

metadata = pd.read_parquet(
    PROCESSED_DIR / "ungdc_preprocessed.parquet",
    columns=["filename", "country_code", "year", "raw_text"]
)

print(f"Loaded {len(metadata):,} speeches")

Loaded 11,141 speeches


## NER

In [2]:
entity_labels = {
    "GPE",       # Countries, cities, geopolitical entities
    "LOC",       # Geographic locations
    "ORG",       # Organizations and institutions
    "PERSON",    # People
    "NORP"       # Nationalities, religious and political groups
}

if RUN_NER or not ENTITIES_PATH.exists():
    import spacy

    # Disable components not needed for named entity recognition
    nlp = spacy.load(
        "en_core_web_sm",
        disable=["parser", "tagger", "lemmatizer", "attribute_ruler"]
    )
    print("Loaded spaCy model:", nlp.meta["name"], nlp.meta["version"])

    entity_records = []

    texts = (
        (text if isinstance(text, str) else "", i)
        for i, text in enumerate(metadata["raw_text"])
    )

    for count, (doc, i) in enumerate(
        nlp.pipe(texts, as_tuples=True, batch_size=64),
        start=1
    ):
        row = metadata.iloc[i]

        for ent in doc.ents:
            if ent.label_ in entity_labels:
                entity_records.append({
                    "filename": row["filename"],
                    "country_code": row["country_code"],
                    "year": row["year"],
                    "entity_text": ent.text,
                    "entity_label": ent.label_,
                    "start_char": ent.start_char,
                    "end_char": ent.end_char
                })

        if count % 250 == 0 or count == len(metadata):
            print(f"Processed {count:,}/{len(metadata):,} documents "
                  f"({count / len(metadata) * 100:.1f}%)")

    entities = pd.DataFrame(entity_records)

    print(f"\nExtracted {len(entities):,} entity mentions")
    print(f"Unique entity strings: {entities['entity_text'].nunique():,}")

    entities.to_parquet(ENTITIES_PATH, index=False)
else:
    entities = pd.read_parquet(ENTITIES_PATH)
    print(f"Loaded cached entities from {ENTITIES_PATH.name}")

# Cached results must correspond to the current corpus
assert set(entities["filename"]) <= set(metadata["filename"])
print(f"Speeches with at least one entity: {entities['filename'].nunique():,} / {len(metadata):,}")
print(f"Entity mentions: {len(entities):,}")
print(f"Unique entity strings: {entities['entity_text'].nunique():,}")

Loaded cached entities from ungdc_entities.parquet


Speeches with at least one entity: 11,141 / 11,141
Entity mentions: 1,299,321
Unique entity strings: 66,325


In [3]:
# Inspect entity categories
display(
    entities["entity_label"]
    .value_counts()
    .rename_axis("entity_type")
    .reset_index(name="mentions")
)

,entity_type,mentions
0,ORG,504983
1,GPE,436247
2,NORP,155798
3,LOC,117835
4,PERSON,84458


In [4]:
# Explore most frequently mentioned entities
entity_summary = (
    entities
    .groupby(["entity_text", "entity_label"])
    .agg(
        mentions=("entity_text", "size"),
        speeches=("filename", "nunique")
    )
    .reset_index()
    .sort_values("mentions", ascending=False)
)

display(entity_summary.head(50))

,entity_text,entity_label,mentions,speeches
65683,the United Nations,ORG,85091,10743
32314,States,GPE,29376,7863
52425,the General Assembly,ORG,28517,9291
39194,United Nations,ORG,27263,8489
1257,Africa,LOC,24336,6075
3382,Assembly,ORG,19925,7040
25254,Organization,ORG,19351,6772
16818,Israel,GPE,16069,4015
62805,the Security Council,ORG,15539,6564
32122,State,ORG,14468,6336


In [5]:
# Save entity summary
entity_summary.to_parquet(
    PROCESSED_DIR / "ungdc_entity_summary.parquet",
    index=False
)

print("Saved NER outputs to:", PROCESSED_DIR)

Saved NER outputs to: /Users/johncreighton/CS6220/International Political Discourse Mining/data/processed
